In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

pd.set_option('display.max_columns', None)

In [ ]:
import sqlite3

# 1. TABLA DIMENSIONAL: OPERARIOS Y TURNOS
operarios_data = {
    'id_operario': ['OP-01', 'OP-02', 'OP-03', 'OP-04', 'OP-05'],
    'nombre': ['Marcos', 'Lucia', 'Diego', 'Ana', 'Carlos'],
    'turno': ['Mañana', 'Mañana', 'Tarde', 'Tarde', 'Noche']
}
df_operarios = pd.DataFrame(operarios_data)

# 2. TABLA DE HECHOS: TICKETS DE DESPACHO (Conectado al Proyecto 1)
# Nos conectamos a la base que generó el notebook 01 y traemos los id_venta
# REALES que sobrevivieron a la limpieza (después de sacar duplicados y
# cantidades inválidas). Antes acá se generaba un rango independiente
# range(1000, 4000), que no coincidía exactamente con los id_venta de
# ventas_limpias y hacía que el JOIN entre ventas y logística perdiera filas.
conn = sqlite3.connect('ecommerce_portfolio.db')
ventas_ids = pd.read_sql_query("SELECT id_venta FROM ventas_limpias", conn)['id_venta'].tolist()
conn.close()

np.random.seed(42)
random.seed(42)
num_ventas = len(ventas_ids)  # un ticket de despacho por cada venta real

df_logistica = pd.DataFrame({
    'id_ticket': [f'TK-{i:05d}' for i in range(1, num_ventas + 1)],
    'id_venta': ventas_ids,  # CLAVE: ids reales, ya no un rango independiente
    'id_operario': [random.choice(df_operarios['id_operario']) for _ in range(num_ventas)],
    
    # Tiempo del ciclo de preparación del pedido en minutos (con algo de ruido)
    'tiempo_preparacion_min': np.random.normal(loc=25.0, scale=8.0, size=num_ventas).round(1),
    
    # Estado del ticket de resolución
    'estado_despacho': np.random.choice(
        ['Completado a Tiempo', 'Con Demora', 'Cancelado - Falta Stock', None], 
        p=[0.75, 0.15, 0.05, 0.05], 
        size=num_ventas
    )
})

# Agregamos errores intencionales para limpiar: tiempos negativos o muy extremos
df_logistica.loc[random.sample(range(num_ventas), min(20, num_ventas)), 'tiempo_preparacion_min'] = -15.0
df_logistica.loc[random.sample(range(num_ventas), min(30, num_ventas)), 'tiempo_preparacion_min'] = np.nan

print("¡Dataset de logística generado y conectado con los id_venta reales de ventas_limpias!")

In [ ]:
# 1. Limpieza de errores en tiempos de preparación (quitamos nulos y negativos)
df_logistica = df_logistica.dropna(subset=['tiempo_preparacion_min'])
df_logistica = df_logistica[df_logistica['tiempo_preparacion_min'] > 0]

# 2. Limpieza de estados nulos
df_logistica['estado_despacho'] = df_logistica['estado_despacho'].fillna('Pendiente de Revisión')

# 3. Unificar datos de los operarios al ticket
df_ops_master = df_logistica.merge(df_operarios, on='id_operario', how='left')

# 4. KPI CLAVE: Semaforización del SLA (Acuerdo de Nivel de Servicio)
# Si el pedido se preparó en menos de 30 mins es "Verde", menos de 45 es "Amarillo", sino "Rojo"
def calcular_sla(tiempo):
    if tiempo <= 30:
        return 'Verde (Óptimo)'
    elif tiempo <= 45:
        return 'Amarillo (Alerta)'
    else:
        return 'Rojo (Crítico)'

df_ops_master['kpi_semaforo_sla'] = df_ops_master['tiempo_preparacion_min'].apply(calcular_sla)

df_ops_master.head()

In [ ]:
import sqlite3

# Nos conectamos a la MISMA base de datos del E-commerce
conn = sqlite3.connect('ecommerce_portfolio.db')

# Guardamos la tabla de logística
df_ops_master.to_sql('logistica_operaciones', conn, if_exists='replace', index=False)

print("¡Tabla de logística integrada al motor SQL!")

In [ ]:
# Query que cruza VENTAS (Proyecto 1) con LOGÍSTICA (Proyecto 2)
query_dashboard = """
    SELECT 
        l.turno AS Turno_Operativo,
        l.estado_despacho AS Estado_Ticket,
        COUNT(l.id_ticket) AS Volumen_Tickets,
        ROUND(AVG(l.tiempo_preparacion_min), 2) AS Promedio_Ciclo_Minutos,
        SUM(v.ingreso_total) AS Capital_Involucrado_USD
    FROM logistica_operaciones l
    JOIN ventas_limpias v ON l.id_venta = v.id_venta
    GROUP BY l.turno, l.estado_despacho
    ORDER BY l.turno, Volumen_Tickets DESC;
"""

df_kpis = pd.read_sql_query(query_dashboard, conn)
display(df_kpis)

conn.close()

In [ ]:
import sqlite3
import pandas as pd

# 1. Nos conectamos a la base
conn = sqlite3.connect('ecommerce_portfolio.db')

# 2. Hacemos una query que traiga TODO unido (Ventas + Logística)
query_power_bi = """
    SELECT 
        v.id_venta,
        v.fecha,
        v.nombre AS producto,
        v.categoria,
        v.cantidad,
        v.ingreso_total,
        v.metodo_pago,
        l.id_operario,
        l.turno,
        l.tiempo_preparacion_min,
        l.estado_despacho
    FROM ventas_limpias v
    LEFT JOIN logistica_operaciones l ON v.id_venta = l.id_venta;
"""

df_export = pd.read_sql_query(query_power_bi, conn)
conn.close()

# 3. Exportamos a un archivo físico listo para Power BI
df_export.to_csv('dataset_power_bi.csv', index=False, encoding='utf-8')
print("¡Archivo 'dataset_power_bi.csv' generado con éxito! Listo para importar en Power BI.")